In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc matplotlib numpy qiskit-addon-cutting
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

*Stima di utilizzo: un minuto su un processore Eagle (NOTA: Questa è solo una stima. Il vostro tempo di esecuzione potrebbe variare.)*

## Contesto

Il circuit-knitting è un termine ombrello che racchiude vari metodi di partizionamento di un circuito in più sottocircuiti più piccoli che coinvolgono meno porte e/o qubit. Ognuno dei sottocircuiti può essere eseguito indipendentemente e il risultato finale si ottiene mediante un post-processing classico sull'esito di ciascun sottocircuito. Questa tecnica è accessibile nell'[addon Qiskit per il taglio di circuiti](https://qiskit.github.io/qiskit-addon-cutting/index.html), una spiegazione dettagliata della tecnica è fornita nella [documentazione](https://qiskit.github.io/qiskit-addon-cutting/explanation/index.html) insieme ad altro [materiale introduttivo](https://qiskit.github.io/qiskit-addon-cutting/tutorials/index.html).

Questo notebook tratta un metodo chiamato <b>taglio di fili</b> in cui il circuito viene partizionato lungo il filo [\[1\], \[2\]](#references). Si noti che il partizionamento è semplice nei circuiti classici poiché l'esito nel punto di partizionamento può essere determinato in modo deterministico ed è 0 oppure 1. Tuttavia, lo stato del qubit nel punto del taglio è, in generale, uno stato misto. Pertanto, ogni sottocircuito deve essere misurato più volte in basi diverse (di solito un insieme tomograficamente completo di basi come la base di Pauli [\[3\], \[4\]](#references) e di conseguenza preparato nel suo autostato. La figura seguente (<i>per gentile concessione di: Tesi di Dottorato, Ritajit Majumdar</i>) mostra un esempio di taglio di fili per uno stato GHZ a 4 qubit in tre sottocircuiti. Qui $M_j$ denota un insieme di basi (di solito Pauli X, Y e Z) e $P_i$ denota un insieme di autostati (di solito $|0\rangle$, $|1\rangle$, $|+\rangle$ e $|+i\rangle$).

![wc-1.png](../docs/images/tutorials/wire-cutting-to-improve-performance/0ce8857b-7f5f-400e-8536-6a496c724d50.avif)
![wc-2.png](../docs/images/tutorials/wire-cutting-to-improve-performance/cbce4455-4794-4c81-8630-3e3993e1b29f.avif)

Poiché ogni sottocircuito ha meno qubit e/o porte, ci si aspetta che sia meno suscettibile al rumore. Questo notebook mostra un esempio in cui questo metodo può essere utilizzato per sopprimere efficacemente il rumore nel sistema.

## Requisiti

Prima di iniziare questo tutorial, assicuratevi di avere installato quanto segue:

- Qiskit SDK v2.0 o successivo, con supporto per la [visualizzazione](https://docs.quantum.ibm.com/api/qiskit/visualization)
- Qiskit Runtime v0.22 o successivo ( `pip install qiskit-ibm-runtime` )
- Addon Qiskit per il taglio di circuiti v0.9.0 o successivo (`pip install qiskit-addon-cutting`)

Considereremo un circuito di Many Body Localization (MBL) per questo notebook. Il circuito MBL è un circuito hardware-efficient ed è parametrizzato da due parametri $\theta$ e $\vec{\phi}$. Quando $\theta$ è impostato a $0$ e lo stato iniziale è preparato in $|0\rangle$ per tutti i qubit, il valore di aspettazione ideale di $\langle Z_i \rangle$ è $+1$ per ogni sito di qubit $i$ indipendentemente dai valori di $\vec{\phi}$. Potete verificare maggiori dettagli sui circuiti MBL in <a href="https://arxiv.org/abs/2307.07552">questo articolo</a>.

## Configurazione

In [1]:
import numpy as np
import matplotlib.pyplot as plt

from qiskit.circuit import Parameter, ParameterVector, QuantumCircuit
from qiskit.quantum_info import PauliList, SparsePauliOp
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_aer import AerSimulator
from qiskit.result import sampled_expectation_value

from qiskit_addon_cutting.instructions import CutWire
from qiskit_addon_cutting import (
    cut_wires,
    expand_observables,
    partition_problem,
    generate_cutting_experiments,
    reconstruct_expectation_values,
)

from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import SamplerV2, Batch

## Parte I. Esempio su piccola scala

### Passo 1: Mappare gli input classici a un problema quantistico

Inizialmente costruiamo un circuito template senza alcun valore di parametro specifico. Forniamo anche dei segnaposto, chiamati `CutWire`, per annotare la posizione dei tagli. Per l'esempio su piccola scala consideriamo un circuito MBL a 10 qubit.

In [2]:
class MBLChainCircuit(QuantumCircuit):
    def __init__(
        self, num_qubits: int, depth: int, use_cut: bool = False
    ) -> None:
        super().__init__(
            num_qubits, name=f"MBLChainCircuit<{num_qubits}, {depth}>"
        )
        evolution = MBLChainEvolution(num_qubits, depth, use_cut)
        self.compose(evolution, inplace=True)


class MBLChainEvolution(QuantumCircuit):
    def __init__(self, num_qubits: int, depth: int, use_cut) -> None:
        super().__init__(
            num_qubits, name=f"MBLChainEvolution<{num_qubits}, {depth}>"
        )

        theta = Parameter("θ")
        phis = ParameterVector("φ", num_qubits)

        for layer in range(depth):
            layer_parity = layer % 2
            # print("layer parity", layer_parity)
            for qubit in range(layer_parity, num_qubits - 1, 2):
                # print(qubit)
                self.cz(qubit, qubit + 1)
                self.u(theta, 0, np.pi, qubit)
                self.u(theta, 0, np.pi, qubit + 1)
                if (
                    use_cut
                    and layer_parity == 0
                    and (
                        qubit == num_qubits // 2 - 1
                        or qubit == num_qubits // 2
                    )
                ):
                    self.append(CutWire(), [num_qubits // 2])
                if use_cut and layer < depth - 1 and layer_parity == 1:
                    if qubit == num_qubits // 2:
                        self.append(CutWire(), [qubit])
            for qubit in range(num_qubits):
                self.p(phis[qubit], qubit)

In [3]:
num_qubits = 10
depth = 2
mbl = MBLChainCircuit(num_qubits, depth)
mbl.draw("mpl", fold=-1)

<Image src="../docs/images/tutorials/wire-cutting/extracted-outputs/a3debf65-06df-4277-933e-14b6f6170756-0.avif" alt="Output of the previous code cell" />

We calculate the average expectation value $O = \frac{1}{n} \sum_i Z_i$ over all qubits for $\theta = 0$. Since the ideal expectation value of $\langle Z_i \rangle = 1$ $\forall$ $i$, the ideal expectation value of $O$ is also $1$. The parameters $\phi$ are selected randomly.

In [4]:
np.random.seed(42)
phis = list(np.random.rand(mbl.num_parameters - 1))
theta = [0]
params = theta + phis

Ora annotiamo il circuito per il taglio inserendo il **CutWire** appropriato per creare due tagli approssimativamente uguali. Impostiamo `use_cut=True` nella funzione e permettiamo che annoti dopo $\frac{n}{2}$ qubit, dove $n$ è il numero di qubit nel circuito originale.

In [5]:
mbl_cut = MBLChainCircuit(num_qubits, depth, use_cut=True)
mbl_cut.assign_parameters(params, inplace=True)
mbl_cut.draw("mpl", fold=-1)

<Image src="../docs/images/tutorials/wire-cutting/extracted-outputs/5208e0a8-0.avif" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/tutorials/wire-cutting/extracted-outputs/31844134-514b-46ea-85f9-133e432f053f-0.avif)

### Passo 2: Ottimizzare il problema per l'esecuzione sull'hardware quantistico
Successivamente tagliamo il circuito in due sottocircuiti più piccoli. Per questo esempio, ci limitiamo a soli 2 sottocircuiti. Per questo, utilizziamo l'<a href="https://qiskit.github.io/qiskit-addon-cutting/">Addon Qiskit: Circuit Cutting</a>.
#### Tagliare il circuito in sottocircuiti più piccoli
Tagliare il filo in un punto aumenta il conteggio dei qubit di uno. Oltre al qubit originale, c'è ora un qubit aggiuntivo come segnaposto per il circuito dopo il taglio. L'immagine seguente fornisce una rappresentazione:

![wc-4.png](../docs/images/tutorials/wire-cutting-to-improve-performance/dfc5f923-e507-4873-888e-d90e1618be3a.avif)

Questo Addon utilizza la funzione `cut_wires` per tenere conto dei qubit aggiuntivi che emergono a causa del taglio.

In [6]:
mbl_move = cut_wires(mbl_cut)
mbl_move.draw("mpl", fold=-1)

<Image src="../docs/images/tutorials/wire-cutting/extracted-outputs/1834cb22-0.avif" alt="Output of the previous code cell" />

#### Creare ed espandere le osservabili
Ora costruiamo l'osservabile $M_z = \frac{1}{n}\sum_{i=1}^n \langle Z_i \rangle$. Poiché l'esito ideale di $\langle Z_i \rangle$ per ogni $i$ è $+1$, l'esito ideale di $M_z$ è anch'esso $+1$.

In [7]:
observable = PauliList(
    ["I" * i + "Z" + "I" * (num_qubits - i - 1) for i in range(num_qubits)]
)
observable

PauliList(['ZIIIIIIIII', 'IZIIIIIIII', 'IIZIIIIIII', 'IIIZIIIIII',
           'IIIIZIIIII', 'IIIIIZIIII', 'IIIIIIZIII', 'IIIIIIIZII',
           'IIIIIIIIZI', 'IIIIIIIIIZ'])

In [8]:
new_obs = expand_observables(observable, mbl, mbl_move)
new_obs

PauliList(['ZIIIIIIIIII', 'IZIIIIIIIII', 'IIZIIIIIIII', 'IIIZIIIIIII',
           'IIIIZIIIIII', 'IIIIIIZIIII', 'IIIIIIIZIII', 'IIIIIIIIZII',
           'IIIIIIIIIZI', 'IIIIIIIIIIZ'])

Tuttavia, si noti che il numero di qubit nel circuito è aumentato dopo l'inserimento delle operazioni virtuali `Move` a 2 qubit dopo il taglio. Pertanto, dobbiamo espandere anche le osservabili inserendo identità per adattarsi al circuito corrente.

In [12]:
partitioned_problem = partition_problem(circuit=mbl_move, observables=new_obs)
subcircuits = partitioned_problem.subcircuits
subobservables = partitioned_problem.subobservables

Here we visualize the two subcircuits:

In [13]:
subcircuits[0].draw("mpl", fold=-1)

<Image src="../docs/images/tutorials/wire-cutting/extracted-outputs/1b0e779f-0.avif" alt="Output of the previous code cell" />

In [14]:
subcircuits[1].draw("mpl", fold=-1)

<Image src="../docs/images/tutorials/wire-cutting/extracted-outputs/3c802f28-0.avif" alt="Output of the previous code cell" />

Visualizziamo i sottocircuiti

In [15]:
M_z = SparsePauliOp(
    ["I" * i + "Z" + "I" * (num_qubits - i - 1) for i in range(num_qubits)],
    coeffs=[1 / num_qubits] * num_qubits,
)

As discussed before, for each cut the upstream circuit must be measured in a Pauli basis, and the downstream circuit must be prepared in the eigenstate of the basis. The function `generate_cutting_experiments` creates all of these necessary circuits and the coefficients associated with each circuit required for reconstruction. Find more detail in [this paper](https://journals.aps.org/prl/abstract/10.1103/PhysRevLett.125.150504).

In [16]:
subexperiments, coefficients = generate_cutting_experiments(
    circuits=subcircuits,
    observables=subobservables,
    num_samples=np.inf,
)

#### Transpile the circuits onto the backend

For the first example involving only simulation, we transpile the circuit into the basis gate set of the backend:

In [17]:
service = QiskitRuntimeService()
backend = service.least_busy(
    operational=True, simulator=False, min_num_qubits=133
)

print(backend)

<IBMBackend('ibm_fez')>


![Output of the previous code cell](../docs/images/tutorials/wire-cutting/extracted-outputs/35920640-76e8-4af6-a252-ee6a22e9c26a-0.avif)

Anche le osservabili sono state partizionate per adattarsi ai sottocircuiti

In [20]:
pm_basis = generate_preset_pass_manager(
    optimization_level=2, basis_gates=backend.configuration().basis_gates
)
basis_subexperiments = {
    label: pm_basis.run(partition_subexpts)
    for label, partition_subexpts in subexperiments.items()
}

In [21]:
sampler = SamplerV2(mode=AerSimulator())
jobs = {
    label: sampler.run(subsystem_subexpts, shots=2**12)
    for label, subsystem_subexpts in basis_subexperiments.items()
}

Si noti che ogni sottocircuito porta a un numero di campioni. La ricostruzione tiene conto dell'esito di ciascuno di questi campioni. Ognuno di questi campioni è chiamato `subexperiment`.
L'estensione dell'osservabile utilizzando l'operazione `Move` richiede una struttura dati `PauliList`. Possiamo anche creare l'osservabile $M_z$ nella struttura dati più generica `SparsePauliOp` che sarà utile in seguito durante la ricostruzione dei subexperiment.

In [22]:
# Retrieve results
results = {label: job.result() for label, job in jobs.items()}

In [23]:
reconstructed_expval_terms = reconstruct_expectation_values(
    results,
    coefficients,
    subobservables,
)
reconstructed_expval = np.dot(reconstructed_expval_terms, M_z.coeffs).real
reconstructed_expval

np.float64(0.9953821063041687)

In [32]:
methods = [
    "Uncut",
    "Wire cut",
]
values = [
    1,
    reconstructed_expval,
]  # since the ideal expectation value in noiseless simulation is +1

ax = plt.gca()
plt.bar(methods, values, color="#a56eff", width=0.4, edgecolor="#8a3ffc")
ax.set_ylabel(r"$M_Z$", fontsize=12)

Text(0, 0.5, '$M_Z$')

<Image src="../docs/images/tutorials/wire-cutting/extracted-outputs/fb5f955a-1.avif" alt="Output of the previous code cell" />

Vediamo due esempi in cui i qubit tagliati vengono misurati in due basi diverse. Per primo, viene misurato nella normale base Z, e successivamente viene misurato nella base X.

In [ ]:
num_qubits = 60
depth = 2

# construct the circuit
mbl = MBLChainCircuit(num_qubits, depth)

# create parameters
phis = list(np.random.rand(mbl.num_parameters - 1))
theta = [0]
params = theta + phis

# construct the cut circuit
mbl_cut = MBLChainCircuit(num_qubits, depth, use_cut=True)
mbl_cut.assign_parameters(params, inplace=True)
mbl_move = cut_wires(mbl_cut)

# Define observable and expand to account for the wire cut
observable = PauliList(
    ["I" * i + "Z" + "I" * (num_qubits - i - 1) for i in range(num_qubits)]
)
new_obs = expand_observables(observable, mbl, mbl_move)

# Construct a SparsePauliOp version of the observable for later use in reconstruction
M_z = SparsePauliOp(
    ["I" * i + "Z" + "I" * (num_qubits - i - 1) for i in range(num_qubits)],
    coeffs=[1 / num_qubits] * num_qubits,
)

# Partition the circuit and get subcircuits and subobservables
partitioned_problem = partition_problem(circuit=mbl_move, observables=new_obs)
subcircuits = partitioned_problem.subcircuits
subobservables = partitioned_problem.subobservables

# Obtain subexperiments and coefficients
subexperiments, coefficients = generate_cutting_experiments(
    circuits=subcircuits,
    observables=subobservables,
    num_samples=np.inf,
)

# Transpile the subexperiments to the backend
pm = generate_preset_pass_manager(optimization_level=2, backend=backend)
isa_subexperiments = {
    label: pm.run(partition_subexpts)
    for label, partition_subexpts in subexperiments.items()
}

# Execute the subexperiments and retrieve results
with Batch(backend=backend) as batch:
    sampler = SamplerV2(mode=batch)
    sampler.options.environment.job_tags = ["TUT_WC"]
    jobs = {
        label: sampler.run(subsystem_subexpts, shots=2**12)
        for label, subsystem_subexpts in isa_subexperiments.items()
    }
results = {label: job.result() for label, job in jobs.items()}

# Reconstruct the expectation value of the original observable
reconstructed_expval_terms = reconstruct_expectation_values(
    results,
    coefficients,
    subobservables,
)
reconstructed_expval = np.dot(reconstructed_expval_terms, M_z.coeffs).real

# Compute the uncut circuit to obtain the noisy expectation value for comparison
sampler = SamplerV2(mode=backend)
sampler.options.environment.job_tags = ["TUT_WC"]

if mbl.num_clbits == 0:
    mbl.measure_all()
isa_mbl = pm.run(mbl)

pub = (isa_mbl, params)
uncut_job = sampler.run([pub])

uncut_counts = uncut_job.result()[0].data.meas.get_counts()
uncut_expval = sampled_expectation_value(uncut_counts, M_z)

# visualize the results
ax = plt.gca()
methods = ["uncut", "cut"]
values = [uncut_expval, reconstructed_expval]

plt.bar(methods, values, color="#a56eff", width=0.4, edgecolor="#8a3ffc")
plt.axhline(y=1, color="k", linestyle="--")
plt.text(0.3, 0.95, "Exact result")
plt.show()

<Image src="../docs/images/tutorials/wire-cutting/extracted-outputs/37834c72-0.avif" alt="Output of the previous code cell" />

In [29]:
uncut_expval

0.9202473958333336

![Output of the previous code cell](../docs/images/tutorials/wire-cutting/extracted-outputs/987547e4-296a-41e4-ad82-41f4139a87a0-0.avif)

#### Traspilare ogni subexperiment

Attualmente dobbiamo traspilare i nostri circuiti prima di inviarli per l'esecuzione. Pertanto, traspilaremo prima ogni circuito nei subexperiment.